# Follow-up experiments for paper 2: cells C (ResNet-18) and E (Kaggle T4 x2)

Runs the remaining experiments on **Kaggle T4 x2** (the benchmark runs of cells C, D and E were made on Kaggle with `cu128`):
1. **E1c**: the 2 missing configurations on ResNet-18 (cell C), `adabelief-dw` and `sfadamw-l2` (LR search with 30-epoch runs + 5 seeds × 100 epochs; ~7.4 GPU hours ≈ **3.7 hours** on 2 GPUs).
2. **E5**: seeds 3, 4 for the weight-decay values **next to the best one** in cell E (~0.5 GPU hours ≈ **15 minutes**).
3. **E6**: check of $\rho_\ell$ against the actual update in cell E (5 configurations of Section V-C × 2 seeds; ~0.7 GPU hours ≈ **20 minutes**).
4. **E5b**: seeds 3, 4 for the remaining values in cell E (~0.4 GPU hours ≈ **12 minutes**).
5. **E6c**: weight norms and actual updates for the 4 benchmark configurations of the 2×2 design on ResNet-18 (`adam`, `adamw`, `sfadamw`, `adabelief`, seeds 0–1; ~4.8 GPU hours ≈ **2.4 hours**).

In total ~**7 hours** of wall time on 2 GPUs, i.e. **one Commit session** (Kaggle limit ~12 hours).

**Settings (right panel):** Accelerator `GPU T4 x2` · Internet `On` · Persistence `Files only`.

**Input:** (1) the code dataset `adam-bench-v4` (upload `adam-bench-v4.zip` to Kaggle Datasets); (2) if available, a CIFAR-10 dataset (otherwise the notebook downloads it before starting the 2 workers); (3) to continue in a second session: the Output of this notebook.

**Run:** the first time set `RUN_CHECKS = True` and Run All interactively until the selftest passes; then set `RUN_CHECKS = False` and **Save Version → Save & Run All (Commit)**.

In [ ]:
# === Cell 1. Parameters ===
CELLS         = ['C', 'E']
EXPS          = ['E1c', 'E5', 'E6', 'E5b', 'E6c']   # in order of priority
RUN_CHECKS    = False    # True the first time, to run the checks
MAX_HOURS     = 10.5     # start no new run after this many hours
TIME_BUDGET_H = 11.25    # force a stop if still running (Kaggle ends a session at ~12 hours)
N_WORKERS     = 2

WORK    = '/kaggle/working'
CODE    = f'{WORK}/adam-bench-v4'
DATA    = f'{WORK}/data'
RESULTS = f'{WORK}/results_exp'

In [ ]:
# === Cell 2. Copy the code and the data (downloaded once, so the 2 workers do not collide); restore the results of a previous session ===
import glob, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

runner = [h for h in glob.glob('/kaggle/input/**/run_exp.py', recursive=True) if '/results' not in h]
if not runner:
    raise SystemExit('run_exp.py NOT found in the Input; add the adam-bench-v4 dataset.')
if os.path.exists(CODE):
    shutil.rmtree(CODE)
shutil.copytree(os.path.dirname(runner[0]), CODE,
                ignore=shutil.ignore_patterns('results*', 'data', '__pycache__'))
print('Code:', os.path.dirname(runner[0]))

os.makedirs(DATA, exist_ok=True)
dst = os.path.join(DATA, 'cifar-10-batches-py')
if not os.path.exists(dst):
    dirs = [d for d in glob.glob('/kaggle/input/**/cifar-10-batches-py', recursive=True) if os.path.isdir(d)]
    tars = glob.glob('/kaggle/input/**/cifar-10-python.tar.gz', recursive=True)
    if dirs:
        shutil.copytree(dirs[0], dst); print('Data: copied', dirs[0])
    elif tars:
        shutil.copy(tars[0], DATA)
        with tarfile.open(tars[0]) as t: t.extractall(DATA)
        print('Data: extracted', tars[0])
    else:
        print('Data: downloading CIFAR-10 once before starting the 2 workers...')
        import torchvision
        torchvision.datasets.CIFAR10(DATA, train=True, download=True)
        torchvision.datasets.CIFAR10(DATA, train=False, download=True)

def count_done(d):
    return len(glob.glob(os.path.join(d, '**', 'result.json'), recursive=True))
if os.path.isdir(RESULTS):
    print('Using the existing results_exp:', count_done(RESULTS), 'runs')
else:
    cands = [c for c in glob.glob('/kaggle/input/**/results_exp', recursive=True) if os.path.isdir(c)]
    if cands:
        best = max(cands, key=count_done)
        shutil.copytree(best, RESULTS)
        print('Restored from', best, ':', count_done(RESULTS), 'runs')
    else:
        print('Starting from scratch')

In [ ]:
# === Cell 3. Use the torch + cuDNN of the benchmark (the runner is a subprocess, so NO restart is needed) ===
import subprocess, sys
WANT = {'B': ('2.11.0', '0.26.0', 'cu130', 92700),
        'C': ('2.11.0', '0.26.0', 'cu128', 91900),
        'D': ('2.11.0', '0.26.0', 'cu128', 91900),
        'E': ('2.11.0', '0.26.0', 'cu128', 91900)}
tv, vv, cu, cudnn_want = WANT[CELLS[0]]
want = f'{tv}+{cu}'

def probe():
    out = subprocess.run([sys.executable, '-c',
                          'import torch; print(torch.__version__); print(torch.backends.cudnn.version()); '
                          'print("|".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())))'],
                         capture_output=True, text=True).stdout.splitlines()
    return out[0], int(out[1]), out[2] if len(out) > 2 else ''

have, cudnn, gpus = probe()
print('torch installed:', have, '| needed:', want)
if have != want:
    print('Reinstalling torch to match the benchmark (a few minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'torch=={tv}+{cu}', f'torchvision=={vv}+{cu}',
                           '--index-url', f'https://download.pytorch.org/whl/{cu}'])
    have, cudnn, gpus = probe()
    print('torch after install:', have)
assert have == want, f'Could not install torch {want}; stopping.'

if cudnn != cudnn_want:
    pkg = 'nvidia-cudnn-cu13' if cu.startswith('cu13') else 'nvidia-cudnn-cu12'
    spec = f'{pkg}=={cudnn_want // 10000}.{cudnn_want // 100 % 100}.*'
    print(f'cuDNN is {cudnn}, needed {cudnn_want} -> installing {spec}')
    subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', spec])
    have, cudnn, gpus = probe()
print('torch', have, '| cuDNN', cudnn, '(needed', cudnn_want, ') | GPU:', gpus)
assert 'T4' in gpus, 'Not a T4 GPU; switch the runtime/accelerator to T4.'
FORCE_ENV = cudnn != cudnn_want
if FORCE_ENV:
    print('WARNING: could not match cuDNN. Continuing with --force-env: torch and GPU match, '
          'only the cuDNN patch level differs; results are statistically equivalent but may not match digit for digit.')
else:
    print('Environment matches the benchmark.')

In [ ]:
# === Cell 4. Run on several GPUs within a time budget ===
import signal, threading
os.chdir(CODE)
T0 = time.time()
n_gpu = int(subprocess.run([sys.executable, '-c', 'import torch; print(torch.cuda.device_count())'],
                           capture_output=True, text=True).stdout.strip() or 0)
N = min(N_WORKERS, n_gpu)
if N == 0:
    raise SystemExit('No GPU. Settings → Accelerator → GPU T4 x2.')
print('Using', N, 'GPUs')

def run_parallel(args, n, budget_s, log):
    os.makedirs(os.path.dirname(log), exist_ok=True)
    f = open(log, 'a'); lk = threading.Lock(); procs = []
    def pump(p, k):
        for line in p.stdout:
            if not line.startswith('['):
                line = f'[w{k}] ' + line
            with lk:
                print(line, end='', flush=True); f.write(line); f.flush()
    for k in range(n):
        e = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k), PYTHONUNBUFFERED='1')
        cmd = [sys.executable, 'run_exp.py'] + args + ['--worker', str(k), '--workers', str(n)]
        print(f'$ CUDA_VISIBLE_DEVICES={k}', ' '.join(cmd), flush=True)
        p = subprocess.Popen(cmd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, start_new_session=True)
        th = threading.Thread(target=pump, args=(p, k), daemon=True); th.start()
        procs.append((p, th)); time.sleep(5)
    while any(p.poll() is None for p, _ in procs):
        if time.time() - T0 > budget_s:
            print('\n>>> Time budget used up; stopping the workers (checkpoints are safe).', flush=True)
            for p, _ in procs:
                if p.poll() is None: os.killpg(p.pid, signal.SIGTERM)
            for p, _ in procs:
                try: p.wait(120)
                except subprocess.TimeoutExpired: os.killpg(p.pid, signal.SIGKILL)
            break
        time.sleep(10)
    rcs = [p.wait() for p, _ in procs]
    for _, th in procs: th.join(5)
    f.close()
    print(f'[exit {rcs}] after {(time.time()-T0)/3600:.2f} hours', flush=True)
    return max(rcs, key=abs)

In [ ]:
# === Cell 5. Checks (first time) ===
if RUN_CHECKS:
    assert subprocess.call([sys.executable, 'run_exp.py', '--selftest', '--out', '/kaggle/tmp_st']) == 0, 'run_exp SELFTEST failed'

In [ ]:
# === Cell 6. Run ===
subprocess.call([sys.executable, 'run_exp.py', '--plan', '--cells', *CELLS, '--exp', *EXPS])
if not RUN_CHECKS:
    args = ['--exp', *EXPS, '--cells', *CELLS, '--out', RESULTS, '--data', DATA, '--max-hours', str(MAX_HOURS)] + (['--force-env'] if FORCE_ENV else [])
    rc = run_parallel(args, N, TIME_BUDGET_H * 3600, os.path.join(RESULTS, 'kaggle_log.txt'))
    if rc == 2:
        print('!!! Environment/configuration error; see the log above.')

In [ ]:
# === Cell 7. Remove checkpoints, show progress, package the results ===
if os.path.isdir(RESULTS):
    for res in glob.glob(os.path.join(RESULTS, '**', 'result.json'), recursive=True):
        for ck in Path(os.path.dirname(res)).glob('*.pt*'):
            ck.unlink()
    subprocess.call([sys.executable, 'run_exp.py', '--status', '--cells', *CELLS, '--exp', *EXPS, '--out', RESULTS])
    print('Created', shutil.make_archive(f'{WORK}/results_exp_CE_v4', 'zip', WORK, 'results_exp'))
shutil.rmtree(CODE, ignore_errors=True)
shutil.rmtree(DATA, ignore_errors=True)